# [Step 1 - HuggingFace models] The model zoo, hosted or local

> **MLCourse - Agentic AI - Chat Models and Providers**

> Stage in the capstone: the generate stage plus a preview of module 07 - the same
> HuggingFace Hub that hosts the text models you try here also hosts the embedding
> models your final RAG chatbot will use, so learning to pick a repo id now pays
> twice.

## What you'll learn

- What the HuggingFace Hub is and how its hosted inference lets you call community models over HTTPS.
- A practical checklist for choosing a `repo_id` (task, license, size, gating, chat template).
- How to call `HuggingFaceEndpoint` with a guarded `HUGGINGFACEHUB_API_TOKEN`.
- The keyless LOCAL alternative and why interface parity makes swapping trivial.
- Honest guidance on when hosted HF inference is the right tool versus Ollama or Groq.

In [1]:
# --- Standard library imports -------------------------------------------------
import os                # Reads environment variables after load_dotenv fills them.
from pathlib import Path # Locates the track root folder for .env loading.

# --- Third-party imports ------------------------------------------------------
from dotenv import load_dotenv  # Loads KEY=value pairs from the track-level .env file.

# Shared walk-up block: find 03_agentic_ai/ and load its gitignored .env so provider
# keys become available in os.environ no matter which subfolder the notebook runs from.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

# HuggingFace's token env var has an unusual long name - read it ONCE, reuse everywhere.
HF_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")

# Jupyter plotting magic inside try/except keeps this file valid pure Python outside IPython.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("HF token present:", bool(HF_TOKEN))   # Boolean only - never echo secret values.

HF token present: False


## 1. What the Hub actually offers

HuggingFace is "GitHub for machine learning": over a million public model repos,
each with a **model card** documenting task, training data, license, and usage.
Two very different ways to use those models:

1. **Hosted inference** (this notebook): send a prompt over HTTPS to a copy of the
   model running on HuggingFace's infrastructure or partner providers. No download,
   no GPU, works from any machine - including thin laptops and CI servers.
2. **Local download**: fetch weights and run them yourself (that is essentially what
   Ollama automates for chat models - see notebook 01).

Hosted access needs a free account token from hf.co/settings/tokens stored as
`HUGGINGFACEHUB_API_KEY`... almost. The env var name is the historical
**`HUGGINGFACEHUB_API_TOKEN`** - a classic gotcha, so double-check it in `.env.example`.

## 2. Choosing a repo_id without regret

A `repo_id` looks like `"org/model-name"` - for example
`"meta-llama/Llama-3.2-3B-Instruct"`. Before committing to one, check five things
on its model card:

1. **Task fit** - is it a text-generation / conversational model, not an image or audio model?
2. **Size vs endpoint capacity** - 1B-8B parameters are safe hosted choices; huge models may queue or fail.
3. **License and gating** - gated repos (most Meta Llamas) require clicking "accept"
   on the model page with your HF account BEFORE the API will serve you.
4. **Chat template** - "-Instruct" / "-Chat" variants know how to follow instructions;
   base models just continue text and will ramble.
5. **Activity** - recent downloads/updates hint the repo is maintained and served reliably.

Good beginner-friendly picks: `meta-llama/Llama-3.2-3B-Instruct` (accept license once)
or fully ungated alternatives such as `microsoft/Phi-3-mini-4k-instruct` or
`HuggingFaceTB/SmolLM2-1.7B-Instruct`.

> **Common pitfall:** a fresh 403/"not authorized" error usually means the repo is
> GATED and you skipped the one-click license acceptance on its web page - not that
> your token is wrong.

## 3. Calling `HuggingFaceEndpoint`

`langchain-huggingface` ships `HuggingFaceEndpoint`: point it at a repo_id plus your
token and it returns generated text. One API subtlety matters: unlike the chat classes
from notebooks 01-02, an endpoint object is a raw TEXT-IN / TEXT-OUT completion model -
`.invoke()` hands you back a plain string, NOT an AIMessage with `.content`. For full
chat-role semantics LangChain wraps it in `ChatHuggingFace`, shown right after.

Both calls sit inside guards: token missing prints setup help; runtime errors (cold
endpoint, gated repo, network) print a friendly skip instead of a traceback.

In [2]:
REPO_ID = "meta-llama/Llama-3.2-3B-Instruct"   # Swap for an ungated pick if you skip licensing.

if not HF_TOKEN:
    # Missing-token path: exact remediation steps, then move on gracefully.
    print("[demo skipped] Add HUGGINGFACEHUB_API_TOKEN to 03_agentic_ai/.env (see .env.example)")
else:
    from langchain_huggingface import HuggingFaceEndpoint

    llm = HuggingFaceEndpoint(
        repo_id=REPO_ID,                    # Which Hub repo to hit (see checklist above).
        huggingfacehub_api_token=HF_TOKEN,  # Explicit beats env-magic when teaching.
        temperature=0.3,                    # Same creativity dial as every other provider.
        max_new_tokens=128,                 # HF names the output cap differently - note it!
    )
    try:
        result = llm.invoke("Explain embeddings in two sentences.")
        print(result)                       # Plain STRING here - endpoints are completion-style.
    except Exception:
        print("[demo skipped] Hosted inference unavailable - cold endpoint, gated repo, or network.")

[demo skipped] Add HUGGINGFACEHUB_API_TOKEN to 03_agentic_ai/.env (see .env.example)


## 4. Chat semantics via `ChatHuggingFace`

If you want the familiar chat interface - typed messages in, `AIMessage` out, working
with system prompts - wrap the endpoint in `ChatHuggingFace`. This wrapper is what you
would pipe into LCEL chains later in the track, exactly like ChatOllama and ChatGroq.

In [3]:
if not HF_TOKEN:
    print("[demo skipped] Add HUGGINGFACEHUB_API_TOKEN to 03_agentic_ai/.env (see .env.example)")
else:
    from langchain_core.messages import SystemMessage, HumanMessage
    from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

    endpoint = HuggingFaceEndpoint(          # Reuse the same underlying engine as section 3.
        repo_id=REPO_ID,
        huggingfacehub_api_token=HF_TOKEN,
        max_new_tokens=128,
    )
    chat_model = ChatHuggingFace(llm=endpoint)   # Thin wrapper adding chat-role handling.

    try:
        reply = chat_model.invoke([
            SystemMessage(content="You answer in exactly two sentences."),   # Now system roles work.
            HumanMessage(content="What is a vector database?"),
        ])
        print(type(reply).__name__, "->", reply.content[:200])   # AIMessage, truncated for tidiness.
    except Exception:
        print("[demo skipped] Hosted inference unavailable - cold endpoint, gated repo, or network.")

[demo skipped] Add HUGGINGFACEHUB_API_TOKEN to 03_agentic_ai/.env (see .env.example)


## 5. The keyless local alternative

Everything sections 3-4 did required a token and outbound internet. Notebook 01's
stack needs neither - and because LangChain unifies the interface, swapping engines is
a one-line change at construction time while every downstream line stays identical:

| | Hosted endpoint (above) | Local Ollama (below) |
|---|---|---|
| Key needed | `HUGGINGFACEHUB_API_TOKEN` | none |
| Internet | required | not after the initial pull |
| Model choice | millions of repos | whatever you pulled |
| Privacy | prompt leaves the machine | nothing leaves the machine |

In [4]:
from langchain_ollama import ChatOllama

local_llm = ChatOllama(model="llama3.2", temperature=0.3)   # Same temperature for a fair feel.

try:
    # IDENTICAL question to section 3 so you can compare answer styles side by side.
    print(local_llm.invoke("Explain embeddings in two sentences.").content)
except Exception:
    print("[demo skipped] Start Ollama, then run once: ollama pull llama3.2")

[demo skipped] Start Ollama, then run once: ollama pull llama3.2


## 6. When does hosted HF make sense?

Reach for `HuggingFaceEndpoint` when:

- You have **no capable GPU** but want a specific open model beyond what Ollama serves easily.
- You need a **niche or fine-tuned checkpoint** (domain adapters, community fine-tunes)
  that no local tool ships pre-packaged.
- You are running in **CI or constrained environments** where installing model weights
  per-job is impractical but an HTTPS call is fine.
- You want to **audition many candidate models** quickly before committing to hosting one.

Prefer local (Ollama) or fast free-tier cloud (Groq) when latency consistency, quota
freedom, or strict privacy matter more than checkpoint variety.

> **Pro tip:** hosted endpoints can be COLD on first request (the serving container
> boots on demand). If a first call hangs, retry once before assuming breakage - and
> keep `timeout` configured in production code.

## Summary & key takeaways

- The Hub is the catalog of open ML models; hosted inference lets you CALL any of them
  over HTTPS without downloads or GPUs.
- Picking a `repo_id` is a five-point check: task, size, license/gating, chat template,
  activity - and gated repos need a one-time license acceptance on their web page.
- `HuggingFaceEndpoint` is text-in/text-out (invoke returns a plain string); wrap it in
  `ChatHuggingFace` for system/human role support like the other chat classes.
- Interface parity means the keyless local alternative differs by ONE constructor line -
  a preview of the provider-swap pattern that notebook 04 makes explicit.
- Env var spelling matters: it is `HUGGINGFACEHUB_API_TOKEN`, loaded from the shared
  track-level `.env`.